# Sample-level PCA by Clinical Outcome

Pseudobulks the 24 single-cell samples (sum of raw counts per `batch`), attaches
`clinical_outcome` / `years_of_survival` from `clinical_info.csv`, and plots PC1 vs PC2
colored by clinical outcome — styled after
`bulk_rna_data/outputs/images/clinical_outcome/pca_by_ClinicalOutcome_annotated.svg`.

This notebook is read-only with respect to the source AnnData: `adata` is only ever
read from, never assigned to, and is never written back to disk.

In [ ]:
import os
# Project data root. Override with the ACC_DATA_ROOT environment variable.
ACC_DATA_ROOT = os.environ.get("ACC_DATA_ROOT", os.path.expanduser("~/acc_data"))


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt

In [ ]:
# Read-only for the rest of this notebook: adata is never assigned to or written back to disk.
adata = sc.read_h5ad(f'{ACC_DATA_ROOT}/outputs/h5ad_files/inprogress_3.h5ad')
adata

In [ ]:
def pseudobulk_by_sample(adata, sample_col="batch", counts_layer="counts", min_cells_per_sample=20):
    """Sum per-cell raw counts into a samples x genes AnnData (one row per sample)."""
    X = adata.layers[counts_layer]
    if not sparse.issparse(X):
        X = sparse.csr_matrix(X)

    obs = adata.obs[[sample_col]].astype(str)
    cell_counts = obs[sample_col].value_counts()
    keep_samples = cell_counts[cell_counts >= min_cells_per_sample].index
    keep_mask = obs[sample_col].isin(keep_samples).values

    obs_f = obs.loc[keep_mask]
    X_f = X[keep_mask, :]

    sample_ids = obs_f[sample_col].values
    uniq_samples = pd.Index(pd.unique(sample_ids), name=sample_col)
    sample_to_row = pd.Series(np.arange(len(uniq_samples)), index=uniq_samples)

    row = sample_to_row.loc[sample_ids].to_numpy()
    col = np.arange(obs_f.shape[0])
    data = np.ones(obs_f.shape[0], dtype=np.int64)
    A = sparse.csr_matrix((data, (row, col)), shape=(len(uniq_samples), obs_f.shape[0]))

    pbX = (A @ X_f).astype(np.int64)  # samples x genes

    pb = ad.AnnData(X=pbX, obs=pd.DataFrame(index=uniq_samples), var=adata.var.copy())
    pb.obs["n_cells"] = cell_counts.loc[uniq_samples].values
    return pb

# Pseudobulk the 24 samples (min_cells_per_sample well below the smallest sample, P03 @ 1052 cells)
pb = pseudobulk_by_sample(adata, sample_col="batch", counts_layer="counts", min_cells_per_sample=20)
pb

In [ ]:
# Attach clinical metadata from the CSV directly onto the pseudobulk obs (adata itself is never touched)
clinical_df = pd.read_csv(f'{ACC_DATA_ROOT}/clinical_info.csv').set_index('Sample ID')

pb.obs['years_of_survival'] = pb.obs.index.map(clinical_df['Years'])
pb.obs['clinical_outcome'] = pb.obs.index.map(clinical_df['Clinical outcome']).astype('category')
pb.obs['clinical_outcome'] = pb.obs['clinical_outcome'].cat.reorder_categories(['Good', 'Poor'])

# Keep the Good/Poor color mapping consistent with the rest of this project's single-cell plots
pb.uns['clinical_outcome_colors'] = adata.uns['outcome_colors']

pb.obs

In [ ]:
# Normalize (CPM), log-transform, scale, then PCA
sc.pp.normalize_total(pb, target_sum=1e6)
sc.pp.log1p(pb)
sc.pp.scale(pb, max_value=10)
sc.tl.pca(pb, n_comps=10, svd_solver="arpack")

In [ ]:
# Plot PC1 vs PC2 colored by clinical outcome, styled after
# bulk_rna_data/outputs/images/clinical_outcome/pca_by_ClinicalOutcome_annotated.svg
var_ratio = pb.uns["pca"]["variance_ratio"]
fig = sc.pl.pca(
    pb, color="clinical_outcome", size=130, legend_loc="right margin",
    alpha=0.9, show=False, return_fig=True,
    title=f"Sample-level PCA by Clinical Outcome  [log1p(CPM); {pb.n_vars} genes; n={pb.n_obs} samples]"
)
ax = fig.axes[0]
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}% variance)")

fig.savefig(
    f'{ACC_DATA_ROOT}/outputs/images/pca_by_clinical_outcome.svg',
    format='svg', bbox_inches='tight'
)
fig